# Golden Dragon WOK ERP – Analitik Modul 3

Pengembangan fitur berbasis paradigma fungsional untuk ERP restoran Cina **Golden Dragon WOK**. Fitur ini memanfaatkan konsep list comprehension, nested list, map, filter, reduce, dan rekursi.

In [13]:
golden_menu = [
    {
        "kode": "GDW-01",
        "nama": "Xiao Long Bao Premium",
        "kategori": "Dim Sum",
        "harga": 38000,
        "stok": 28,
        "rating": [5, 5, 4, 5],
        "bahan": ["Kaldu ayam", "Daging ayam cincang", "Kulit pangsit"]
    },
    {
        "kode": "GDW-02",
        "nama": "Kung Pao Lobster",
        "kategori": "Main Course",
        "harga": 68000,
        "stok": 14,
        "rating": [5, 4, 5, 5, 4],
        "bahan": ["Lobster", "Kacang mete", "Cabai sichuan"]
    },
    {
        "kode": "GDW-03",
        "nama": "Chongqing Mala Noodles",
        "kategori": "Main Course",
        "harga": 52000,
        "stok": 20,
        "rating": [4, 5, 4, 4],
        "bahan": ["Mie telur", "Minyak cabai", "Sichuan pepper"]
    },
    {
        "kode": "GDW-04",
        "nama": "Crispy Duck Mantou",
        "kategori": "Signature",
        "harga": 75000,
        "stok": 10,
        "rating": [5, 5, 5, 4, 5],
        "bahan": ["Bebek panggang", "Mantou", "Saus plum"]
    },
    {
        "kode": "GDW-05",
        "nama": "Osmanthus Honey Tea",
        "kategori": "Beverage",
        "harga": 28000,
        "stok": 40,
        "rating": [4, 4, 5, 4],
        "bahan": ["Bunga osmanthus", "Madu", "Goji berry"]
    }
]

riwayat_pesanan = [
    {
        "nomor": "PO-GDW-001",
        "status": "selesai",
        "bundles": [
            [
                {"kode": "GDW-01", "jumlah": 3},
                {"kode": "GDW-05", "jumlah": 2}
            ],
            [
                {"kode": "GDW-03", "jumlah": 1}
            ]
        ]
    },
    {
        "nomor": "PO-GDW-002",
        "status": "selesai",
        "bundles": [
            [
                {"kode": "GDW-02", "jumlah": 2}
            ],
            [
                {"kode": "GDW-04", "jumlah": 1}
            ]
        ]
    },
    {
        "nomor": "PO-GDW-003",
        "status": "proses",
        "bundles": [
            [
                {"kode": "GDW-05", "jumlah": 3}
            ],
            [
                {"kode": "GDW-01", "jumlah": 1},
                {"kode": "GDW-03", "jumlah": 2}
            ]
        ]
    },
    {
        "nomor": "PO-GDW-004",
        "status": "selesai",
        "bundles": [
            [
                {"kode": "GDW-04", "jumlah": 1},
                {"kode": "GDW-01", "jumlah": 2}
            ]
        ]
    }
]

### Struktur Data Operasional
Dataset `golden_menu` disusun sebagai daftar kamus untuk menjaga keterbacaan atribut menu dan memudahkan komposisi fungsi. Setiap entri memuat daftar bahan (`bahan`) sehingga contoh nested list bisa diolah lebih lanjut. Sementara itu `riwayat_pesanan` menggunakan susunan `bundles` bertingkat yang merepresentasikan paket pesanan multi-menu—struktur ini penting agar fungsi rekursif dapat menelusuri setiap item tanpa mutasi.

In [ ]:
from functools import reduce
import json

def beri_penjelasan(alasan):
    def dekorator(fungsi):
        def pembungkus(*args, **kwargs):
            return {
                "fungsi": fungsi.__name__,
                "alasan": alasan,
                "output": fungsi(*args, **kwargs)
            }
        return pembungkus
    return dekorator

def susun_lookup_harga(menu):
    return {item["kode"]: item["harga"] for item in menu}

def kelompok_kategori(menu):
    kategori_terurut = sorted({item["kategori"] for item in menu})
    return [
        [kategori, [item["nama"] for item in menu if item["kategori"] == kategori]]
        for kategori in kategori_terurut
    ]

def jumlahkan_rating(total, nilai):
    return total + nilai


def hitung_rata_rating(item):
    total = reduce(jumlahkan_rating, item["rating"], 0)
    rata = round(total / len(item["rating"]), 2) if item["rating"] else 0
    return {**item, "rata_rating": rata}


def padukan_rating(menu):
    return list(map(hitung_rata_rating, menu))

def seleksi_promo(menu_dengan_rating, batas_rating, batas_stok):
    def memenuhi(item):
        return item["rata_rating"] >= batas_rating and item["stok"] <= batas_stok

    return list(filter(memenuhi, menu_dengan_rating))

def flatten_pesanan(bundles):
    if isinstance(bundles, dict):
        return [bundles]
    if not bundles:
        return []
    kepala, *ekor = bundles
    return flatten_pesanan(kepala) + flatten_pesanan(ekor)

def total_porsi(order):
    return sum(item["jumlah"] for item in flatten_pesanan(order["bundles"]))


def hitung_tagihan(order, lookup_harga):
    detail = flatten_pesanan(order["bundles"])

    def akumulasi(total, item):
        return total + lookup_harga[item["kode"]] * item["jumlah"]

    return reduce(akumulasi, detail, 0)


def total_pendapatan(orders, lookup_harga):
    def tambah(total, order):
        return total + hitung_tagihan(order, lookup_harga)
    return reduce(tambah, orders, 0)


def rangkum_pesanan(orders, lookup_harga):
    return [
        {
            "nomor": order["nomor"],
            "status": order["status"],
            "total_porsi": total_porsi(order),
            "total_tagihan": hitung_tagihan(order, lookup_harga)
        }
        for order in orders
    ]


@beri_penjelasan("List comprehension dan nested list dipakai supaya kategori menu langsung memuat daftar hidangan sehingga modul inventori mudah membaca hierarki.")
def laporan_kategori(menu):
    return kelompok_kategori(menu)

@beri_penjelasan("Map dan reduce menambah rata-rata rating tanpa mengubah struktur asli menu sehingga analitik kualitas tetap pure.")
def laporan_rating(menu):
    return padukan_rating(menu)


@beri_penjelasan("Filter menyaring kandidat promo berdasarkan stok rendah namun rating tinggi untuk mendukung keputusan marketing.")
def laporan_promo(menu_dengan_rating, batas_rating, batas_stok):
    return seleksi_promo(menu_dengan_rating, batas_rating, batas_stok)


@beri_penjelasan("Rekursi flatten_pesanan memastikan paket pesanan bertingkat terurai rapi ketika menghitung porsi dan omzet.")
def laporan_rangkuman_pesanan(orders, lookup_harga):
    return rangkum_pesanan(orders, lookup_harga)

### Arsitektur Fungsi Deklaratif
- `beri_penjelasan` bertindak sebagai dekorator untuk menambahkan catatan penggunaan setiap fitur secara konsisten di hasil laporan.
- Fungsi utilitas (`susun_lookup_harga`, `kelompok_kategori`) memanfaatkan list comprehension serta set comprehension guna menata kembali data tanpa loop eksplisit.
- `padukan_rating`, `seleksi_promo`, dan `total_pendapatan` menunjukkan kombinasi `map`, `filter`, dan `reduce` yang menjaga kemurnian data sambil menghitung metrik penting.
- `flatten_pesanan` menerapkan rekursi untuk meratakan struktur paket bertingkat, sehingga fungsi agregasi lain dapat tetap sederhana dan deklaratif.

In [15]:
lookup_harga = susun_lookup_harga(golden_menu)
kategori_report = laporan_kategori(golden_menu)
rating_report = laporan_rating(golden_menu)
menu_dengan_rating = rating_report["output"]
promo_report = laporan_promo(menu_dengan_rating, batas_rating=4.6, batas_stok=20)
ringkasan_report = laporan_rangkuman_pesanan(riwayat_pesanan, lookup_harga)
pemasukan_total = total_pendapatan(riwayat_pesanan, lookup_harga)

laporan_final = {
    "kategori_menu": kategori_report,
    "menu_dengan_rating": rating_report,
    "kandidat_promo": promo_report,
    "ringkasan_pesanan": ringkasan_report,
    "total_pendapatan": pemasukan_total
}

print(json.dumps(laporan_final, indent=4))

{
    "kategori_menu": {
        "fungsi": "laporan_kategori",
        "alasan": "List comprehension dan nested list dipakai supaya kategori menu langsung memuat daftar hidangan sehingga modul inventori mudah membaca hierarki.",
        "output": [
            [
                "Beverage",
                [
                    "Osmanthus Honey Tea"
                ]
            ],
            [
                "Dim Sum",
                [
                    "Xiao Long Bao Premium"
                ]
            ],
            [
                "Main Course",
                [
                    "Kung Pao Lobster",
                    "Chongqing Mala Noodles"
                ]
            ],
            [
                "Signature",
                [
                    "Crispy Duck Mantou"
                ]
            ]
        ]
    },
    "menu_dengan_rating": {
        "fungsi": "laporan_rating",
        "alasan": "Map dan reduce menambah rata-rata rating tanpa mengubah struktur 

### Interpretasi Laporan
Sel eksekusi laporan mengumpulkan seluruh keluaran ke dalam kamus `laporan_final`. Struktur ini memudahkan modul lain membaca:
- `kategori_menu`: hierarki menu berbasis kategori untuk inventori dan kurasi menu.
- `menu_dengan_rating`: data enriched dengan rata-rata rating sebagai dasar analitik kualitas.
- `kandidat_promo`: daftar menu yang memenuhi syarat promo kilat berdasarkan stok dan rating.
- `ringkasan_pesanan`: ringkasan tiap order yang sudah diratakan sehingga total porsi dan tagihan siap dipantau.
- `total_pendapatan`: akumulasi omzet dari riwayat pesanan aktif.

### Ringkasan Implementasi Konsep
- **List comprehension & nested list**: menyusun kembali kategori menu sekaligus mempertahankan hubungan menu-bahan untuk kebutuhan inventori.
- **Map & reduce**: memperkaya data menu dengan rata-rata rating serta menghitung omzet secara fungsional.
- **Filter**: memberi shortlist menu berpotensi promo berdasarkan kriteria bisnis yang dapat diubah secara parametrik.
- **Rekursi**: menjamin struktur pesanan bertingkat tetap dapat dihitung tanpa mengorbankan paradigma fungsional.
- **Dekorator**: memastikan setiap fungsi laporan mengembalikan penjelasan penggunaan sehingga dokumentasi teknis melekat pada hasil eksekusi.